In [1]:
"""
PEFT_Sentiment_Analysis_on_Movie_Reviews.API

This notebook demonstrates:
1. The native HuggingFace APIs (tokenizer, model, dataset)
2. The PEFT LoRA API (LoraConfig, get_peft_model)
3. How our utils wrapper functions expose a simpler high-level API

This notebook *does not* train a model.
It is purely for understanding the API surface.
"""


'\nPEFT_Sentiment_Analysis_on_Movie_Reviews.API\n\nThis notebook demonstrates:\n1. The native HuggingFace APIs (tokenizer, model, dataset)\n2. The PEFT LoRA API (LoraConfig, get_peft_model)\n3. How our utils wrapper functions expose a simpler high-level API\n\nThis notebook *does not* train a model.\nIt is purely for understanding the API surface.\n'

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import TrainingArguments
from datasets import Dataset

from peft import LoraConfig, TaskType, get_peft_model

from PEFT_Sentiment_Analysis_utils import (
    load_fake_true,
    preprocess_text,
    split_data,
    prepare_hf_dataset,
    load_roberta_lora
)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

sample_text = "Transformers make NLP much easier."

encoded = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=20
)

encoded


{'input_ids': [0, 44820, 268, 146, 234, 21992, 203, 3013, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [4]:
dataset = Dataset.from_dict({
    "text": ["hello world", "this is an example"],
    "label": [0, 1]
})

dataset


Dataset({
    features: ['text', 'label'],
    num_rows: 2
})

In [5]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

model


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [6]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"]
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [8]:
df = load_fake_true("/Users/vikas/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask96_Fall2025_PEFT_Sentiment_Analysis_on_Movie_Reviews/Data/Fake.csv", "/Users/vikas/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask96_Fall2025_PEFT_Sentiment_Analysis_on_Movie_Reviews/Data/True.csv")
df.head()


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [9]:
df = preprocess_text(df)
df[["text", "text_final"]].head()


,text,text_final
0,Donald Trump just couldn t wish all Americans ...,donald trump wish american happy new year leav...
1,House Intelligence Committee Chairman Devin Nu...,house intelligence committee chairman devin nu...
2,"On Friday, it was revealed that former Milwauk...",friday revealed former milwaukee sheriff david...
3,"On Christmas day, Donald Trump announced that ...",christmas day donald trump announced would bac...
4,Pope Francis used his annual Christmas Day mes...,pope francis used annual christmas day message...


In [10]:
train_texts, test_texts, train_labels, test_labels = split_data(df)

train_dataset, test_dataset, tokenizer = prepare_hf_dataset(
    train_texts, test_texts, train_labels, test_labels
)

train_dataset


Map: 100%|██████████| 8980/8980 [00:14<00:00, 603.03 examples/s]


Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 35918
})

In [11]:
model = load_roberta_lora()


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [12]:
print("API demonstration completed successfully.")


API demonstration completed successfully.
